# Indian agriculture 1966-2017: regional gaps

**Notebook 3.** In notebook 2 I looked at India as a whole. Here I break it down by state and district to ask one question: **did the low-yield regions catch up with the high-yield ones?**

I use `crop_data_clean.csv` from notebook 1 again. I focus on rice and wheat because they are the biggest crops and the ones most changed by the Green Revolution. At the end I check the other six crops from notebook 2 to see if the picture is the same.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Loading the data

In [ ]:
crop_data = pd.read_csv("../data/clean/crop_data_clean.csv")
crop_data.shape

In [ ]:
# states and districts in the data
print(crop_data["state"].nunique(), "states")
print(crop_data["dist_code"].nunique(), "districts")

## Two settings I use everywhere

In notebook 1 I said I would leave out tiny areas when comparing yields, because a small mistake in production becomes a huge yield on a tiny area. Here are the two limits:

- **Districts:** area of at least 5 (so 5,000 hectares) in the year I'm looking at.
- **States:** a state only counts as a "major" state for a crop if its average area in 2015-17 is at least 100 (100,000 hectares). Otherwise a state that grows 3,000 hectares of wheat would show up in the wheat ranking next to Punjab.

I picked these numbers by looking at the data, they are a judgement call. The conclusions don't change much if I use 1 instead of 5 for districts (I tried it).

In [ ]:
MIN_DISTRICT_AREA = 5     # 1000 ha
MIN_STATE_AREA = 100      # 1000 ha

main_crops = ["Rice", "Wheat", "Maize", "Sorghum", "Pearl millet", "Chickpea", "Groundnut", "Cotton"]

## State yields

Same idea as notebook 2: add up area and production for all districts of a state, then divide. I don't average district yields, because a small district would count as much as a big one.

In [ ]:
def state_yearly(crop):
    one = crop_data[crop_data["crop"] == crop].dropna(subset=["area", "production"])
    one = one[one["area"] > 0]

    totals = one.groupby(["state", "year"])[["area", "production"]].sum().reset_index()
    totals["crop_yield"] = totals["production"] / totals["area"] * 1000
    return totals


def major_states(totals):
    recent = totals[totals["year"].between(2015, 2017)].groupby("state")["area"].mean()
    return list(recent[recent >= MIN_STATE_AREA].index)


rice_states = state_yearly("Rice")
wheat_states = state_yearly("Wheat")

rice_major = major_states(rice_states)
wheat_major = major_states(wheat_states)

print("rice major states:", len(rice_major))
print("wheat major states:", len(wheat_major))

### Which states are on top now?

In [ ]:
def state_ranking(totals, major):
    recent = totals[totals["year"].between(2015, 2017) & totals["state"].isin(major)]
    return recent.groupby("state")["crop_yield"].mean().sort_values()


fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, name, totals, major in [(axes[0], "Rice", rice_states, rice_major),
                                (axes[1], "Wheat", wheat_states, wheat_major)]:
    ranking = state_ranking(totals, major)
    ax.barh(ranking.index, ranking.values, color="seagreen")
    ax.set_title(name + " yield, average 2015-17")
    ax.set_xlabel("yield (kg per ha)")

plt.tight_layout()
plt.show()

### How the states changed by decade

In [ ]:
def decade_table(totals, major):
    one = totals[totals["state"].isin(major)].copy()
    one["decade"] = (one["year"] // 10 * 10).astype(str) + "s"
    table = one.pivot_table(index="state", columns="decade", values="crop_yield")
    return table.sort_values(table.columns[-1], ascending=False)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(decade_table(rice_states, rice_major), annot=True, fmt=".0f", cmap="YlGn", cbar=False, ax=axes[0])
axes[0].set_title("Rice yield (kg per ha) by decade")

sns.heatmap(decade_table(wheat_states, wheat_major), annot=True, fmt=".0f", cmap="YlGn", cbar=False, ax=axes[1])
axes[1].set_title("Wheat yield (kg per ha) by decade")

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

As in notebook 2, the 1960s column only covers 1966-69 and the 2010s column only 2010-17.

### Best states against worst states

In [ ]:
def plot_top_and_bottom(totals, major, crop):
    ranking = state_ranking(totals, major)
    chosen = list(ranking.index[:3]) + list(ranking.index[-3:])

    one = totals[totals["state"].isin(chosen)].copy()
    one["smooth_yield"] = one.groupby("state")["crop_yield"].transform(
        lambda values: values.rolling(5, center=True).mean())

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=one, x="year", y="smooth_yield", hue="state", hue_order=chosen[::-1], linewidth=2)
    plt.title(crop + " yield: 3 best and 3 worst states in 2015-17 (5 year average)")
    plt.xlabel("year")
    plt.ylabel("yield (kg per ha)")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()


plot_top_and_bottom(rice_states, rice_major, "Rice")

In [ ]:
plot_top_and_bottom(wheat_states, wheat_major, "Wheat")

### Is the gap between states getting smaller?

I compare the best and the worst major state at the start (1966-68) and the end (2015-17). I look at two things: the **ratio** (how many times higher is the best) and the **difference** in kg per hectare. They can tell different stories.

In [ ]:
def gap_between_states(totals, major, crop):
    rows = {}
    for label, first, last in [("1966-68", 1966, 1968), ("2015-17", 2015, 2017)]:
        window = totals[totals["year"].between(first, last) & totals["state"].isin(major)]
        by_state = window.groupby("state")["crop_yield"].mean()
        rows[label] = {"best state": by_state.idxmax(),
                       "best yield": by_state.max(),
                       "worst state": by_state.idxmin(),
                       "worst yield": by_state.min(),
                       "ratio": by_state.max() / by_state.min(),
                       "difference": by_state.max() - by_state.min()}
    table = pd.DataFrame(rows).T
    table.index.name = crop
    return table


pd.concat([gap_between_states(rice_states, rice_major, "Rice"),
           gap_between_states(wheat_states, wheat_major, "Wheat")])

## District spread

States hide a lot, so now the districts. For each year I take all districts with area of at least 5 and measure how spread out their yields are in two ways:

- **Standard deviation** in kg per hectare. This is the absolute gap.
- **Coefficient of variation** (std divided by mean). This is the relative gap, which accounts for the fact that all yields are rising.

If the coefficient of variation falls over time, districts are getting more similar in relative terms.

In [ ]:
def district_spread(crop):
    one = crop_data[(crop_data["crop"] == crop) & (crop_data["area"] >= MIN_DISTRICT_AREA)]
    one = one.dropna(subset=["crop_yield"])

    spread = one.groupby("year")["crop_yield"].agg(["count", "mean", "std"])
    spread["cv"] = spread["std"] / spread["mean"]
    spread["crop"] = crop
    return spread.reset_index()


spread = pd.concat([district_spread("Rice"), district_spread("Wheat")], ignore_index=True)

# smooth lines, because single years jump around with the weather
for column in ["std", "cv"]:
    spread[column + "_smooth"] = spread.groupby("crop")[column].transform(
        lambda values: values.rolling(5, center=True).mean())

spread[spread["year"].isin([1966, 1970, 1980, 1990, 2000, 2010, 2017])].round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.lineplot(data=spread, x="year", y="std_smooth", hue="crop", linewidth=2.5, ax=axes[0])
axes[0].set_title("Absolute gap between districts")
axes[0].set_ylabel("std of district yields (kg per ha)")

sns.lineplot(data=spread, x="year", y="cv_smooth", hue="crop", linewidth=2.5, ax=axes[1])
axes[1].set_title("Relative gap between districts")
axes[1].set_ylabel("coefficient of variation")

for ax in axes:
    ax.set_xlabel("year")

plt.tight_layout()
plt.show()

## Did low-yield districts grow faster?

Here I test catching up directly. For every district I take its yield in 1966-68 (average of 3 years) and in 2015-17, and work out the average yearly growth between the two. The two windows are centred on 1967 and 2016, so that is 49 years.

If low starting yield goes together with fast growth, that means catch-up. A district needs enough area in both windows to be included.

In [ ]:
def district_catch_up(crop):
    one = crop_data[(crop_data["crop"] == crop) & (crop_data["area"] >= MIN_DISTRICT_AREA)]
    one = one.dropna(subset=["crop_yield"])

    start = one[one["year"].between(1966, 1968)].groupby("dist_code")["crop_yield"].mean()
    end = one[one["year"].between(2015, 2017)].groupby("dist_code")["crop_yield"].mean()

    names = crop_data.drop_duplicates("dist_code").set_index("dist_code")[["district", "state"]]

    result = pd.DataFrame({"start_yield": start, "end_yield": end}).dropna()
    result["growth"] = ((result["end_yield"] / result["start_yield"]) ** (1 / 49) - 1) * 100
    return result.join(names)


rice_catch = district_catch_up("Rice")
wheat_catch = district_catch_up("Wheat")

print("districts in the rice comparison:", len(rice_catch))
print("districts in the wheat comparison:", len(wheat_catch))

In [ ]:
def plot_catch_up(catch, crop, ax):
    ax.scatter(catch["start_yield"], catch["growth"], alpha=0.5, color="seagreen")

    # straight line through the points: growth against log of the starting yield
    slope, intercept = np.polyfit(np.log(catch["start_yield"]), catch["growth"], 1)
    x_line = np.linspace(catch["start_yield"].min(), catch["start_yield"].max(), 100)
    ax.plot(x_line, slope * np.log(x_line) + intercept, color="black", linewidth=2)

    correlation = np.corrcoef(np.log(catch["start_yield"]), catch["growth"])[0, 1]
    ax.set_xscale("log")
    ax.set_title(crop + " (correlation " + str(round(correlation, 2)) + ")")
    ax.set_xlabel("yield in 1966-68 (kg per ha, log scale)")
    ax.set_ylabel("growth per year (%)")


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
plot_catch_up(rice_catch, "Rice", axes[0])
plot_catch_up(wheat_catch, "Wheat", axes[1])
plt.suptitle("Starting yield against yearly growth, one dot per district")
plt.tight_layout()
plt.show()

A downward line means districts that started low grew faster. There is one thing to be careful about: a district with a very low starting yield might just have had a bad or badly recorded year, and then it looks like it grew a lot. Using 3 years instead of 1 helps a bit, but not completely.

Let me look at the fastest and slowest districts to see who they are.

In [ ]:
columns = ["district", "state", "start_yield", "end_yield", "growth"]

print("Rice, fastest growing districts")
display(rice_catch.sort_values("growth", ascending=False).head(8)[columns].round(1))

print("Rice, slowest growing districts")
display(rice_catch.sort_values("growth").head(8)[columns].round(1))

In [ ]:
print("Wheat, fastest growing districts")
display(wheat_catch.sort_values("growth", ascending=False).head(8)[columns].round(1))

print("Wheat, slowest growing districts")
display(wheat_catch.sort_values("growth").head(8)[columns].round(1))

## Is it the same for the other crops?

Rice and wheat were pushed by the Green Revolution. The other crops from notebook 2 got far less irrigation and seed research, so I check if their district gaps behave differently. I compare the coefficient of variation of district yields at the start and at the end, using 3-year windows.

In [ ]:
def spread_in_window(crop, first, last):
    one = crop_data[(crop_data["crop"] == crop) & (crop_data["area"] >= MIN_DISTRICT_AREA)]
    one = one[one["year"].between(first, last)].dropna(subset=["crop_yield"])

    # average of each district over the window, so every district counts once
    by_district = one.groupby("dist_code")["crop_yield"].mean()
    return by_district.std() / by_district.mean(), len(by_district)


rows = []
for crop in main_crops:
    cv_start, n_start = spread_in_window(crop, 1966, 1968)
    cv_end, n_end = spread_in_window(crop, 2015, 2017)
    rows.append({"crop": crop, "cv 1966-68": cv_start, "cv 2015-17": cv_end,
                 "districts 1966-68": n_start, "districts 2015-17": n_end})

spread_by_crop = pd.DataFrame(rows).set_index("crop")
spread_by_crop["change"] = spread_by_crop["cv 2015-17"] - spread_by_crop["cv 1966-68"]

spread_by_crop.sort_values("change").round(2)

In [ ]:
spread_by_crop[["cv 1966-68", "cv 2015-17"]].plot(kind="bar", figsize=(10, 5))
plt.title("Relative gap between districts, start and end")
plt.xlabel("")
plt.ylabel("coefficient of variation")
plt.xticks(rotation=0)
plt.show()

## What I found

- **Punjab is the top state for both crops.** In 2015-17 its yield is about 4,160 kg/ha for rice and 4,900 kg/ha for wheat. For wheat Punjab was also first in 1966-68. For rice the top state back then was Karnataka, at about 1,460 kg/ha.
- **The gap between states got smaller in relative terms but bigger in absolute terms.** For rice, the best state was 3.0 times the worst in 1966-68 and 2.5 times in 2015-17. But the difference in kg/ha grew from about 975 to about 2,520. For wheat the ratio went from 5.6 to 5.1 and the difference from about 1,500 to about 3,940 kg/ha. All yields rose, so the same percentage gap means a much bigger number in kg.
- **Districts show the same pattern.** The coefficient of variation of district yields fell from about 0.57 to 0.36 for rice and from 0.54 to 0.36 for wheat. But the standard deviation about doubled (rice 469 to 928, wheat 412 to 1,096 kg/ha). The fall is not smooth: for rice, years like 1980 and 2000 are back close to the starting level.
- **Low-yield districts did grow faster.** The correlation between starting yield and yearly growth is about -0.52 for rice and -0.45 for wheat. Many of the fastest growers are districts in Madhya Pradesh (Rewa, Satna, Sidhi, Jabalpur) that started from only 240 to 370 kg/ha. Districts that were already high, like Ludhiana in wheat, grew slower (1.2% a year). Some of the very low starting values may come from a bad year or poor records, so part of this catch-up could be an artefact.
- **Catch-up was not the same for all crops.** The relative gap between districts shrank for chickpea (-0.11), rice (-0.09) and wheat (-0.07). It stayed about the same for pearl millet and maize, and it grew for groundnut (+0.06), sorghum (+0.13) and cotton (+0.17).

**Limits:** state totals use the fixed 1966 district boundaries, so states created later are built from the old districts. The area limits I used (5 for districts, 100 for states) are my own choice. I looked at yields only and not at why a region grew faster (irrigation, rainfall, soil), because the data has nothing on that.

**Next notebook:** changes in the crop mix, and the weather dips.